In [ ]:
!pip install ultralytics

In [ ]:
from google.colab import drive
drive.mount("./drive", force_remount=True)

## Managing Datasets

In [ ]:
from pathlib import Path
import shutil
import os

In [ ]:
weapon_dataset_url = "https://www.kaggle.com/api/v1/datasets/download/jubaerad/weapons-in-images-segmented-videos"
od_weapon_url = "https://github.com/ari-dasci/OD-WeaponDetection/archive/refs/heads/master.zip"

In [ ]:
print(f"{' Downloading Weapons-in-Images ':=^100}")
!wget {weapon_dataset_url} -P datasets/
!mv datasets/weapons-in-images-segmented-videos datasets/dataset_1.zip

print(f"{'\n Downloading OD Weapon Dataset ':=^100}")
!wget {od_weapon_url} -P datasets/
!mv datasets/master.zip datasets/dataset_2.zip

In [ ]:
# Copying our internal dataset from Drive
!cp /content/drive/MyDrive/dataset/vahagn_terrorist.zip datasets/

In [ ]:
!unzip datasets/dataset_1.zip -d datasets/dataset_1
!unzip datasets/dataset_2.zip -d datasets/dataset_2
!unzip datasets/vahagn_terrorist.zip -d datasets/dataset_3

!rm -rf datasets/dataset_1.zip
!rm -rf datasets/dataset_2.zip
!rm -rf datasets/vahagn_terrorist.zip

## Downloading and Installing Sentry AI

In [ ]:
# !rm -rf ./Sentry-Model/ Model.zip main.zip
# sys.path.pop()

In [ ]:
# Downloading Sentry AI
URL = "https://github.com/vardgeszakaryan/Sentry/archive/refs/heads/main.zip"

In [ ]:
# Downloading
!wget {URL}

# Unzipping
!unzip main.zip

In [ ]:
import sys
import os

# 1. Define the absolute path to the downloaded GitHub repository
# The unzipped folder is 'Sentry-main', so the path should reflect that.
repo_path = r"./Sentry-main/src"

# 2. Add it to sys.path so Python knows to look for modules there
if repo_path not in sys.path:
    sys.path.append(repo_path)

In [ ]:
sys.path

## Filtering folders and images

#### Separating the Kaggle Weapons-in-Images Dataset

In [ ]:
# Initializing inner dataset directories to copy
dataset_dir = Path("datasets")

# OD-dataset from github
knife_detection_od = dataset_dir / "dataset_2" / "OD-WeaponDetection-master" / "Knife_detection"
pistol_detection_od = dataset_dir / "dataset_2" / "OD-WeaponDetection-master" / "Pistol detection"

# Kaggle dataset
test_dataset = dataset_dir / "dataset_1" / "Weapon in Images (Segmented Video)" / "All"
weapoins_in_images = dataset_dir / "dataset_1" / "Weapons-in-Images" / "Weapons-in-Images" # Seperation needed here

# Our dataset
our_dataset = dataset_dir / "dataset_3"

In [ ]:
target_path = Path("dataset_final/processed")
raw_dataset = Path("dataset_final/raw")
processed_dataset = Path("dataset_final/processed")

raw_dataset.mkdir(parents=True, exist_ok=True)
processed_dataset.mkdir(parents=True, exist_ok=True)

In [ ]:
def seperate_txt_jpg(folder_path: Path, target_folder: Path):
  """Separates label (.txt) files from images into dedicated subdirectories.

  Designed for datasets where images and labels live in the same flat folder.

  Args:
    folder_path (pathlib.Path): The source folder containing mixed files.
    target_folder (pathlib.Path): Destination root; `images/` and `labels/` subdirs will be created here.

  Returns:
    None
  """

  images_path = target_folder / "images"
  labels_path = target_folder / "labels"

  labels_path.mkdir(parents=True, exist_ok=True)
  images_path.mkdir(parents=True, exist_ok=True)

  valid_extensions = (".avif", ".bmp", ".dng", ".heic", ".jpg", ".jpeg", ".jp2", ".mpo", ".pfm", ".png", ".tif", ".tiff", ".webp")

  for f_file in folder_path.iterdir():
    print(f_file)
    if f_file.name.lower().endswith(valid_extensions):
      shutil.copy(f_file, images_path / f_file.name)

    elif f_file.name.lower().endswith('.txt'):
      shutil.copy(f_file, labels_path / f_file.name)


In [ ]:
seperate_txt_jpg(weapoins_in_images, (raw_dataset / "dataset_1_weapons"))

#### Converting Knife and Pistol Datasets from Pascal VOC to YOLO

In [ ]:
!pip install pylabel

##### ⚠️ Data Issue: Missing File Extensions in OD-Weapon-Pistol XML Annotations

The XML annotation files in the **OD-Weapon-Pistol dataset** had a structural problem:
the `<filename>` field was missing its file extension entirely.

```xml
<annotation>
  <folder>...</folder>
  <filename>DSC_0002</filename>  <!-- No .jpg extension! -->
  <path>...</path>
  ...
</annotation>
```

Without a valid extension, `pylabel` cannot match annotations to their corresponding images.
The cell below repairs this by scanning the actual image folder, building a filename-to-extension
map, and patching any XML `<filename>` tag that is missing its extension.

In [ ]:
import xml.etree.ElementTree as ET
from pathlib import Path

# --- Configuration ---
# Point these to your specific directories
xml_directory = pistol_detection_od / "xmls"
img_directory = pistol_detection_od / "Weapons"

fixed_count = 0
skipped_count = 0
missing_img_count = 0

print("Mapping image extensions from your image directory...")

# 1. Build a dictionary of your actual images: { 'DSC_0002': '.jpg', 'IMG_05': '.png' }
image_map = {}
# Using rglob to catch images even if they are in subfolders
for img_path in img_directory.rglob('*'):
    # Check if it's a file and looks like an image
    if img_path.is_file() and img_path.suffix.lower() in ['.jpg', '.jpeg', '.png', '.webp']:
        # .stem gets the name without extension (e.g., 'DSC_0002')
        # .suffix gets the extension (e.g., '.jpg')
        image_map[img_path.stem] = img_path.suffix

print(f"Found {len(image_map)} images. Now patching XML files...")

# 2. Patch the XML files
for xml_path in xml_directory.rglob('*.xml'):
    try:
        tree = ET.parse(xml_path)
        root = tree.getroot()
        filename_elem = root.find('filename')

        if filename_elem is not None and filename_elem.text:
            current_filename = filename_elem.text.strip()

            # Check if it is missing a file extension
            if '.' not in current_filename:

                # Check if we have an image that matches this filename
                if current_filename in image_map:
                    correct_extension = image_map[current_filename]
                    # Append the correct extension found in the image folder
                    filename_elem.text = f"{current_filename}{correct_extension}"

                    # Save the changes
                    tree.write(xml_path, encoding='utf-8', xml_declaration=False)
                    fixed_count += 1
                else:
                    print(f"⚠️ Warning: Found XML for '{current_filename}', but no matching image exists in the image folder.")
                    missing_img_count += 1
            else:
                skipped_count += 1

    except Exception as e:
        print(f"❌ Error processing {xml_path.name}: {e}")

print("---")
print("Data repair complete.")
print(f"🔧 Files fixed with correct extensions: {fixed_count}")
print(f"⏭️  Files skipped (already had extensions): {skipped_count}")
print(f"👻 Missing images (XML exists, but no image found): {missing_img_count}")

In [ ]:
from pylabel import importer

# Re-import the datasets after renaming files and updating XML contents
knife_voc = importer.ImportVOC(knife_detection_od/"annotations/", "../Images/", name="knife_detection_od")
pistol_voc = importer.ImportVOC(pistol_detection_od/"xmls/", "../Weapons/", name="pistol_detection_od")

In [ ]:
pistol_voc.visualize.ShowBoundingBoxes(995)

In [ ]:
# Creating Essential directories and files
raw_knife = raw_dataset / "dataset_2_knife"
raw_pistol = raw_dataset / "dataset_3_pistol"

raw_knife.mkdir(parents=True, exist_ok=True)
raw_pistol.mkdir(parents=True, exist_ok=True)

# Exporting labels
knife_voc.export.ExportToYoloV5(
    output_path    = raw_knife / "labels",
    copy_images    = True
)

pistol_voc.export.ExportToYoloV5(
    output_path    = raw_pistol / "labels",
    copy_images    = True
)

''
# Moving .yaml files
# (raw_pistol / "train" / "dataset.yaml").rename(raw_pistol / "dataset.yaml")
# (raw_knife / "train" / "dataset.yaml").rename(raw_knife / "dataset.yaml")

#### Copying test dataset to processed final

In [ ]:
(target_path / "test_dataset").mkdir(parents=True, exist_ok=True)
shutil.copytree(test_dataset, target_path / "test_dataset", dirs_exist_ok=True)

#### Copying our dataset

In [ ]:
(raw_dataset / "dataset_4_ours").mkdir(parents=True, exist_ok=True)
shutil.copytree(our_dataset / "images" / "train", raw_dataset / "dataset_4_ours" / "images", dirs_exist_ok=True)
shutil.copytree(our_dataset / "labels" / "train", raw_dataset / "dataset_4_ours" / "labels", dirs_exist_ok=True)

yaml_content = """
names:
  0: Weapon
path: dataset_final/raw/dataset_4_ours
train: images
"""

dataset_4_yaml = raw_dataset / "dataset_4_ours" / "dataset.yaml"
dataset_4_yaml.touch()
dataset_4_yaml.write_text(yaml_content)

print("4th dataset copied.")
print("dataset.yaml successfully created")

## Dataset Analysis

In [ ]:
# Loading datasets
yaml_config = """
project:
  name: "Final Dataset: Mix 4"
  runs_dir: "runs/"

dataset:
  # Your four datasets
  dataset_1: dataset_final/raw/dataset_1_weapons
  dataset_2: dataset_final/raw/dataset_2_knife
  dataset_3: dataset_final/raw/dataset_3_pistol
  dataset_4: dataset_final/raw/dataset_4_ours
  merged_dir: "dataset_final/processed/merged_datasets/"

  merge_mode: "rebuild"
  rebuild_ratios:
    train: 0.7
    val: 0.15
    test: 0.15

  # REQUIRED: This ensures if any datasets use 1, 2, or 3,
  # they are remapped to 0 so YOLO treats them as the same class.
  class_remap:
    1: 0
    2: 0

training:
  # Model Config
  model_yaml: "yolo26n.pt"
  epochs: 150
  patience: 25

  # Dataset
  batch_size: 16
  imgsz: 1280

  # Devices
  device: "0"
  optimizer: "AdamW"
  workers: 10
  lr0: 0.004

  # Augmentation
  degrees: 10
  fliplr: 0.5
"""

conf_dir = Path("configs")
conf_dir.mkdir(parents=True, exist_ok=True)

config_file = (conf_dir / "model_pipeline.yaml")

config_file.touch(exist_ok=True)
config_file.write_text(yaml_config)

print("Created pipeline_config.yaml")

In [ ]:
# Validating COCO8 Dataset
from sentry_ai.dataset.validate import validate_yolo_dataset

def validate_dataset(dataset_path: str) -> None:
    errors = validate_yolo_dataset(dataset_path)

    if errors:
        print(f"[-] Dataset Validation FAILED with {len(errors)} errors:")

        for i, error in enumerate(errors):
            print(f"{i+1}. {error}")

        print("\n... (showing errors)...")

    else:
        print(f"[+] Validation of {dataset_path} dataset passed.")

In [ ]:
for dataset in raw_dataset.iterdir():
  print(f"{f' Validating {dataset} ':=^75}\n")
  validate_dataset(dataset)

Some validation runs may surface missing-image warnings. This is intentional — a portion of label files have no corresponding image, which teaches the model that not every scene contains a weapon (background negatives).

In [ ]:
from sentry_ai.dataset.audit import audit_yolo_dataset
import json

def audit_dataset(dataset_path: str) -> None:
    stats = audit_yolo_dataset(dataset_path)
    print(json.dumps(stats, indent=4))

In [ ]:
for dataset in raw_dataset.iterdir():
  print(f"{f' Validating {dataset} ':=^75}\n")
  audit_dataset(dataset)

### Merging the final dataset

In [ ]:
from sentry_ai.dataset.merge import merge_datasets
from sentry_ai.config import load_config

config = load_config(config_file)
merge_datasets(config)

In [ ]:
final_dataset = processed_dataset / "merged_datasets"
validate_dataset(final_dataset)
audit_dataset(final_dataset)

In [ ]:
yolo_content = '''
path: dataset_final/processed/merged_datasets
train: images/train
val: images/val
test: images/test

names:
  0: Weapon
'''

yaml_file_path = final_dataset / '/dataset.yaml'
yaml_file_path.write_text(yolo_content)

print(f"Generated dataset.yaml at {yaml_file_path}")

### Analyzing Final Merged dataset

In [ ]:
from sentry_ai.analysis.dataset_analysis import analyze_dataset

analyze_dataset(final_dataset, Path("analyses"))

In [ ]:
from IPython.display import display
from PIL import Image

for file in Path("analyses").iterdir():
  print(f"{f'{file}':=^100}")

  if file.name.endswith("json"):
    print(json.dumps(
        json.load(file.open('r')),
        indent=4
        ))

  else:
    display(Image.open(file))

In [ ]:
from sentry_ai.yolo.train import train_yolo

train_yolo(config)